In [1]:
from email.mime import image

import pandas as pd
import torch
import torch.nn as nn
from PIL import features
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [2]:
delivery_df = pd.DataFrame({
    "trip_distance_miles": [2.5, 4.0, 6.0, 8.5, 10.0, 12.0, 15.0, 18.0],
    "pickup_hour": [8, 9, 17, 18, 7, 12, 16, 20],
    "passenger_count": [1, 1, 2, 2, 1, 3, 2, 1],
    "ride_duration_min": [9.0, 12.0, 18.0, 24.0, 27.0, 31.0, 39.0, 46.0]
})

In [3]:
delivery_df

,trip_distance_miles,pickup_hour,passenger_count,ride_duration_min
0,2.5,8,1,9.0
1,4.0,9,1,12.0
2,6.0,17,2,18.0
3,8.5,18,2,24.0
4,10.0,7,1,27.0
5,12.0,12,3,31.0
6,15.0,16,2,39.0
7,18.0,20,1,46.0


In [4]:
class DeliveryTransform:
    def __call__(self, feature):
        return torch.tensor(feature,dtype=torch.float32)

feature_transform = DeliveryTransform()
sample_features = delivery_df.loc[0, ["trip_distance_miles", "pickup_hour", "passenger_count"]].values
transform_feature = feature_transform(sample_features)

In [5]:
sample_features

array([2.5, 8. , 1. ])

In [6]:
transform_feature

tensor([2.5000, 8.0000, 1.0000])

In [7]:
transform_feature.shape

torch.Size([3])

In [8]:
class DeliveryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.feature_col = ["trip_distance_miles", "pickup_hour", "passenger_count"]
        self.target_col = "ride_duration_min"

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.loc[index]
        features = row[self.feature_col].values
        target = row[self.target_col]

        if self.transform:
            features = self.transform(features)
        else:
            features = torch.tensor(features,dtype=torch.float32)

        target = torch.tensor([target], dtype=torch.float32)
        return features, target

delivery_dataset = DeliveryDataset(delivery_df, transform=feature_transform)

print("DataSet length:", len(delivery_dataset))
print("First sample:")
print(delivery_dataset[0])

DataSet length: 8
First sample:
(tensor([2.5000, 8.0000, 1.0000]), tensor([9.]))


In [9]:
first_feature, first_target = delivery_dataset[0]

print("First feature tensor:")
print(first_feature)
print("shape: ", first_feature.shape)
print("First target tensor:")
print(first_target)
print("shape: ", first_target.shape)

First feature tensor:
tensor([2.5000, 8.0000, 1.0000])
shape:  torch.Size([3])
First target tensor:
tensor([9.])
shape:  torch.Size([1])


In [10]:
delivery_loader = DataLoader(
    delivery_dataset,
    batch_size=3,
    shuffle=True,
)

In [11]:
print(delivery_loader)

In [12]:
for batch_feachers, batch_targets in delivery_loader:
    print("Batch feachers:")
    print(batch_feachers)
    print("shape: ", batch_feachers.shape)
    print("Batch targets:")
    print("shape: ", batch_targets.shape)
    print(batch_targets)
    break

Batch feachers:
tensor([[12.0000, 12.0000,  3.0000],
        [ 2.5000,  8.0000,  1.0000],
        [10.0000,  7.0000,  1.0000]])
shape:  torch.Size([3, 3])
Batch targets:
shape:  torch.Size([3, 1])
tensor([[31.],
        [ 9.],
        [27.]])


In [13]:
import torch.nn.functional as F
from torchvision import datasets, transforms

In [14]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

print(mnist_transform)

Compose(
    ToTensor()
    Normalize(mean=(0.1307,), std=(0.3081,))
)


In [15]:
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=mnist_transform
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=mnist_transform
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Number of training samples:", len(train_dataset))
print("Number of test samples:", len(test_dataset))

Number of training samples: 60000
Number of test samples: 10000


In [16]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("\nFirst label in the batch:", labels[0].item())

Image batch shape: torch.Size([32, 1, 28, 28])
Label batch shape: torch.Size([32])

First label in the batch: 5


In [17]:
sequential_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

print("Sequential model:")
print(sequential_model)

Sequential model:
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=128, bias=True)
  (2): ReLU()
  (3): Linear(in_features=128, out_features=10, bias=True)
)


In [18]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer_1 = nn.Linear(28*28, 128)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        return x

model = MNISTClassifier()
print(model)

MNISTClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer_1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (layer_2): Linear(in_features=128, out_features=10, bias=True)
)


In [19]:
sample_outputs = model(images)

print("Outputs shape from model:", sample_outputs.shape)
print("This means 32 samples, each with 10 class scores.")

Outputs shape from model: torch.Size([32, 10])
This means 32 samples, each with 10 class scores.


In [20]:
class BrokenModel(nn.Module):
    def __init__(self):
        # super().__init__() intentionally skipped
        self.fc = nn.Linear(28*28, 10)

    def forward(self, x):
        return self.fc(x)

try:
    broken_model = BrokenModel()
except Exception as e:
    print("Error when super().__init__() is skipped")
    print(e)

Error when super().__init__() is skipped
cannot assign module before Module.__init__() call


In [21]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters())

In [26]:
model.train()
for batch_idx, (data, labels) in enumerate(train_loader):
    optimizer.zero_grad()
    outputs = model(data)
    loss = loss_function(outputs, labels)
    loss.backward()
    optimizer.step()
    if batch_idx % 200 == 0:
        print(f"Batch {batch_idx} | Loss: {loss.item():.4f}")


Batch 0 | Loss: 0.8781
Batch 200 | Loss: 0.5423
Batch 400 | Loss: 0.4333
Batch 600 | Loss: 0.3239
Batch 800 | Loss: 0.3953
Batch 1000 | Loss: 0.3520
Batch 1200 | Loss: 0.4266
Batch 1400 | Loss: 0.3296
Batch 1600 | Loss: 0.3671
Batch 1800 | Loss: 0.3449


In [27]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, labels in test_loader:
        outputs = model(data)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if total >= 3200:
            break

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 85.75


In [28]:
model.train()

MNISTClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer_1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (layer_2): Linear(in_features=128, out_features=10, bias=True)
)